# Mini-Courier Planner
Integrated capstone implementation: sorting, 0/1 knapsack DP, Dijkstra, nearest-neighbor TSP, and runtime testing.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.data_io import load_parcels, load_graph
from src.sorting import sort_parcels
from src.knapsack import optimize_packing
from src.graph_algorithms import shortest_path
from src.tsp import nearest_neighbor_tsp
import time

In [ ]:
parcels = load_parcels(ROOT/'datasets/parcels.csv')
graph = load_graph(ROOT/'datasets/routes.csv')
ordered = sort_parcels(parcels)
for p in ordered:
    print(p)

In [ ]:
capacity = 20
selected, total_weight, total_value, dp = optimize_packing(ordered, capacity)
print('Selected:', [p.parcel_id for p in selected])
print('Total weight:', total_weight)
print('Total value:', total_value)
print('DP optimum:', dp[-1][-1])

In [ ]:
customers = sorted({p.destination for p in selected})
for customer in customers:
    distance, path = shortest_path(graph, 'Depot', customer)
    print(customer, distance, path)
route, route_distance, legs = nearest_neighbor_tsp(graph, 'Depot', customers)
print('Route:', ' -> '.join(route))
print('Distance:', route_distance)

## Runtime benchmark
The following benchmark repeats the algorithms on prefixes of the parcel dataset.

In [ ]:
sizes = [3, 5, 8, 10, 12]
results = []
for n in sizes:
    subset = parcels[:n]
    t = time.perf_counter(); ordered_n = sort_parcels(subset); sort_t = time.perf_counter()-t
    t = time.perf_counter(); optimize_packing(ordered_n, 20); knap_t = time.perf_counter()-t
    customers_n = sorted({p.destination for p in ordered_n[:min(5,n)]})
    t = time.perf_counter(); nearest_neighbor_tsp(graph, 'Depot', customers_n); route_t = time.perf_counter()-t
    results.append((n, sort_t, knap_t, route_t))
results

In [ ]:
import matplotlib.pyplot as plt
sizes = [r[0] for r in results]
plt.figure(figsize=(8,5))
plt.plot(sizes, [r[1] for r in results], marker='o', label='Sorting')
plt.plot(sizes, [r[2] for r in results], marker='o', label='Knapsack DP')
plt.plot(sizes, [r[3] for r in results], marker='o', label='Route planning')
plt.xlabel('Number of parcels')
plt.ylabel('Runtime (seconds)')
plt.title('Mini-Courier Planner Runtime Comparison')
plt.legend(); plt.grid(True); plt.show()